# staging

In [1]:
# dependencies

#pip install xlsxwriter


In [2]:
# imports and settings

from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils import get_column_letter
from openpyxl import Workbook, load_workbook
from zipfile import BadZipFile
from itertools import product
from typing import Tuple

import pandas as pd
import numpy as np
import difflib
import re
import os

# pandas optional preferences
pd.set_option("display.float_format", "{:,.6f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


In [3]:
# directory & output staging
"""
python file exists one level above "raw_data" folder
"""

DATA_DIR = "_clean_data"
OUT_DIR = "_merge_output"
os.makedirs(OUT_DIR, exist_ok=True)

AUDIT_XLSX = os.path.join(OUT_DIR, "_merge_audit.xlsx")

if not os.path.exists(AUDIT_XLSX):
    wb = Workbook()
    wb.save(AUDIT_XLSX)
else:
    # load if valid; recreate if corrupted
    try:
        wb = load_workbook(AUDIT_XLSX)
    except BadZipFile:
        wb = Workbook()
        wb.save(AUDIT_XLSX)
        wb = load_workbook(AUDIT_XLSX)

    # detect whether workbook contains any cell values
    has_data = False
    for ws in wb.worksheets:
        for row in ws.iter_rows(values_only=True):
            if any(v is not None for v in row):
                has_data = True
                break
        if has_data:
            break

    if has_data:
        for name in list(wb.sheetnames):
            wb.remove(wb[name])
        wb.create_sheet("Sheet")
        wb.active = wb["Sheet"]
        wb.save(AUDIT_XLSX)

csv_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv")]
csv_files


['GSCM 521 - Banner Grade Data v2_clean.csv',
 'GSCM 521 - Canvas Gradebook Data_clean.csv',
 'GSCM 521 - Canvas Interaction Data v2_clean.csv']

In [4]:
# exploration dfs

dfs = {}

for f in csv_files:
    name = f.replace(".csv", "")
    path = os.path.join(DATA_DIR, f)
    dfs[name] = pd.read_csv(path)

# quick sanity check
{key: df.shape for key, df in dfs.items()}


{'GSCM 521 - Banner Grade Data v2_clean': (880, 22),
 'GSCM 521 - Canvas Gradebook Data_clean': (568, 3),
 'GSCM 521 - Canvas Interaction Data v2_clean': (276078, 15)}

In [5]:
# quick head assessments

for name, df in dfs.items():
    print("\n" + name)
    display(df.head(3))



GSCM 521 - Banner Grade Data v2_clean


,subject,course,title,term,random_student,random_id,grade,instructor,crn,reg_status,unduplicated_race,citizenship,overall_ug_credits,overall_gr_credits,credits_in_class_term,1st_term_at_psu,latest_term_with_reg_activity,level,latest_class,major_1_in_class_term,latest_major_1,random_email
0,ACTG,335,Ais And Accounting Analytics,202204,"Aem, Udf",840806293,A,"Kaufman, Matt",10003,RW,White,Citizen,279.500000,NaN,10.000000,201504,202302,NaN,Non-Admit,PBAC--Postbac Account Cert,0000--Undeclared/Not Applicable,UdfAem@pdx.edu
1,ACTG,335,Ais And Accounting Analytics,202204,"Anp, Spo",821380792,C,"Kaufman, Matt",10002,RW,White,Citizen,196.000000,NaN,16.000000,202104,202304,NaN,Non-Admit,ACTG--Business Adm: Accounting,0000--Undeclared/Not Applicable,SpoAnp@pdx.edu
2,ACTG,335,Ais And Accounting Analytics,202204,"Bcq, Zqx",803055564,B-,"Kaufman, Matt",10003,RW,Black or African American,Citizen,150.000000,NaN,12.000000,201904,202302,UG,Senior,ACTG--Business Adm: Accounting,MAL--Bus Adm: Mgmt & Leadership,ZqxBcq@pdx.edu



GSCM 521 - Canvas Gradebook Data_clean


,random_student,sis_id,random_sis_login_id
0,"Tse, Sml",iuybybxh-7b98-4c40-915a-bhuazfvgugyp,SmlTse
1,"Iwu, Xts",hmtdikpg-862d-4948-a099-ilvohytgsfdl,XtsIwu
2,"Kvo, Omb",zzerrowz-23ae-48f1-a83e-lmzanfsqauzd,OmbKvo



GSCM 521 - Canvas Interaction Data v2_clean


,student_id,random_student_name,random_sortable_name,sis_id,section_id,section_name,course_id,course_name,content_type,content_name,times_viewed,times_participated,start_date,first_viewed,last_viewed
0,"200,830,000,000,000,000.000000",Qcx Pyt,"Pyt, Qcx",izbtazwy-1fdc-4d45-98a6-mxwybumlbjgm,"200,830,000,000,000,000.000000",ACTG-335-HB1: Ais And Accounting Analytics,"200,830,000,000,000,000.000000",Ais And Accounting Analytics HB1 Fall 2022,course.files.file,ACTG 335_F22_Kaufman_Syllabus-1.doc,3,0,9/19/2022,9/19/2022 16:29,9/19/2022 17:04
1,"200,830,000,000,000,000.000000",Qcx Pyt,"Pyt, Qcx",izbtazwy-1fdc-4d45-98a6-mxwybumlbjgm,"200,830,000,000,000,000.000000",ACTG-335-HB1: Ais And Accounting Analytics,"200,830,000,000,000,000.000000",Ais And Accounting Analytics HB1 Fall 2022,course.home,Course Home,4,0,9/19/2022,9/19/2022 16:29,9/19/2022 17:17
2,"200,830,000,000,000,000.000000",Qcx Pyt,"Pyt, Qcx",izbtazwy-1fdc-4d45-98a6-mxwybumlbjgm,"200,830,000,000,000,000.000000",ACTG-335-HB1: Ais And Accounting Analytics,"200,830,000,000,000,000.000000",Ais And Accounting Analytics HB1 Fall 2022,course.syllabus,Course Syllabus,3,0,9/19/2022,9/19/2022 16:29,9/19/2022 16:52


# assess for likely join cols

In [6]:
# assess for likely join cols funcs

def _norm_col(s: str) -> str:
    """
    note: Normalizes a column name into a comparable, token-friendly form.
    Lowercases, trims, converts whitespace to underscores, strips non-alphanumerics
    (except underscore), and collapses repeated underscores.
    Intended for generic name similarity only; does not mutate any DataFrame.
    """
    s = str(s).strip().lower()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^a-z0-9_]", "", s)
    s = re.sub(r"_+", "_", s)
    return s

def _col_tokens(s: str) -> set:
    """
    note: Tokenizes a column name into a set of normalized tokens split on underscores.
    Used for overlap-based similarity scoring in join reconnaissance.
    """
    s = _norm_col(s)
    return {t for t in s.split("_") if t}

def _name_similarity(a: str, b: str) -> float:
    """
    note: Computes a generic column-name similarity score in [0, 1].
    Combines token-overlap (Jaccard) and sequence similarity (difflib ratio)
    to rank plausible join key pairs without using domain assumptions.
    """
    ta, tb = _col_tokens(a), _col_tokens(b)
    jacc = (len(ta & tb) / len(ta | tb)) if (ta or tb) else 0.0
    seq = difflib.SequenceMatcher(None, _norm_col(a), _norm_col(b)).ratio()
    return 0.6 * jacc + 0.4 * seq

def _series_profile(s: pd.Series) -> dict:
    """
    note: Computes join-relevant profiling stats for a Series without mutating it.
    Returns null rate, unique counts/ratios, inferred kind (numeric-intlike, numeric-float,
    datetime, stringlike), and average string length (when applicable).
    Used to skip obviously poor join candidates and to enrich recon diagnostics.
    """
    n = len(s)
    nn = int(s.notna().sum())
    null_rate = 1.0 - (nn / n) if n else 1.0
    nun = int(s.nunique(dropna=True))
    distinct_ratio = (nun / nn) if nn else 0.0

    dt = s.dtype
    if pd.api.types.is_datetime64_any_dtype(dt):
        kind = "datetime"
    elif pd.api.types.is_numeric_dtype(dt):
        vals = s.dropna().to_numpy()
        if vals.size:
            frac_intlike = float(np.mean(np.isclose(vals, np.round(vals))))
            kind = "numeric_intlike" if frac_intlike >= 0.995 else "numeric_float"
        else:
            kind = "numeric_float"
    else:
        kind = "stringlike"

    # cheap string length hint (only if object/string)
    avg_strlen = np.nan
    if kind == "stringlike":
        ss = s.dropna().astype("string")
        if len(ss):
            avg_strlen = float(ss.str.len().mean())

    return {
        "null_rate": float(null_rate),
        "distinct_ratio": float(distinct_ratio),
        "n_nonnull": int(nn),
        "n_unique": int(nun),
        "dtype": str(dt),
        "kind": kind,
        "avg_strlen": avg_strlen,
    }

def _coerce_for_compare(s: pd.Series, mode: str) -> pd.Series:
    """
    note: Produces a comparison-ready Series used for join evidence checks.
    mode="raw" returns the Series as-is.
    mode="str" normalizes values to strings (lower/strip), and converts numeric values
    so 123.0 becomes "123" to improve key matching across mixed typing.
    Does not mutate the original Series.
    """
    if mode == "raw":
        return s

    if pd.api.types.is_numeric_dtype(s.dtype):
        v = pd.to_numeric(s, errors="coerce")
        out = v.map(lambda x: "" if pd.isna(x) else (str(int(x)) if float(x).is_integer() else str(x)))
        return out.astype("string")
    return s.astype("string").str.strip().str.lower()

def merge_audit(
    dfs: dict,
    top_k_per_pair: int = 5,
    name_sim_min: float = 0.60,
    sample_rows_left: int = 50000,
    sample_rows_right: int = 200000,
    random_state: int = 7,
    max_null_rate: float = 0.95,
    min_distinct_ratio: float = 0.001,
    max_avg_strlen: float = 80.0,
) -> pd.DataFrame:
    """
    note: Performs generic join reconnaissance across all pairs of tables/columns.
    Uses:
    - name similarity (token overlap + sequence similarity)
    - sampled value-set overlap (unique coverage and row match rate)
    - lightweight profiling filters (null rate, distinct ratio, string length)
    Outputs ranked candidate join pairs with evidence metrics, designed to be
    conservative and fast on large tables.
    """
    rng = np.random.default_rng(random_state)

    tables = list(dfs.keys())

    profiles = {t: {c: _series_profile(dfs[t][c]) for c in dfs[t].columns} for t in tables}
    tokens = {t: {c: _col_tokens(c) for c in dfs[t].columns} for t in tables}

    # right-side caches: (table, col, mode) -> set(values)
    right_set_cache = {}

    def get_right_set(rt: str, rc: str, mode: str) -> set:
        key = (rt, rc, mode)
        if key in right_set_cache:
            return right_set_cache[key]

        s = _coerce_for_compare(dfs[rt][rc], mode).dropna()
        if len(s) > sample_rows_right:
            s = s.sample(n=sample_rows_right, random_state=random_state)
        u = pd.unique(s)
        right_set_cache[key] = set(u.tolist())
        return right_set_cache[key]

    rows = []

    for lt in tables:
        for rt in tables:
            if lt == rt:
                continue

            lcols = list(dfs[lt].columns)
            rcols = list(dfs[rt].columns)

            # build token -> right cols index
            token_to_rcols = {}
            for rc in rcols:
                for tok in tokens[rt][rc]:
                    token_to_rcols.setdefault(tok, set()).add(rc)

            scored = []

            for lc in lcols:
                lp = profiles[lt][lc]

                # generic “not a key” skips
                if lp["null_rate"] > max_null_rate:
                    continue
                if lp["distinct_ratio"] < min_distinct_ratio:
                    continue
                if lp["kind"] == "stringlike" and not np.isnan(lp["avg_strlen"]) and lp["avg_strlen"] > max_avg_strlen:
                    continue

                # candidate right cols: share at least one token
                cand_r = set()
                for tok in tokens[lt][lc]:
                    cand_r |= token_to_rcols.get(tok, set())
                if not cand_r:
                    continue

                # sample left rows once; use for both raw and str
                lraw = _coerce_for_compare(dfs[lt][lc], "raw").dropna()
                if len(lraw) > sample_rows_left:
                    lraw = lraw.sample(n=sample_rows_left, random_state=random_state)
                lstr = _coerce_for_compare(dfs[lt][lc], "str").dropna()
                if len(lstr) > sample_rows_left:
                    lstr = lstr.sample(n=sample_rows_left, random_state=random_state)

                lraw_u = pd.unique(lraw)
                lstr_u = pd.unique(lstr)

                for rc in cand_r:
                    sim = _name_similarity(lc, rc)
                    if sim < name_sim_min:
                        continue

                    rp = profiles[rt][rc]
                    kind_compatible = (lp["kind"] == rp["kind"]) or (
                        lp["kind"].startswith("numeric") and rp["kind"].startswith("numeric")
                    )

                    rraw_set = get_right_set(rt, rc, "raw")
                    rstr_set = get_right_set(rt, rc, "str")

                    # vectorized membership
                    uc_raw = float(np.mean(pd.Series(lraw_u).isin(rraw_set))) if len(lraw_u) else np.nan
                    uc_str = float(np.mean(pd.Series(lstr_u).isin(rstr_set))) if len(lstr_u) else np.nan
                    rm_raw = float(np.mean(lraw.isin(rraw_set))) if len(lraw) else np.nan
                    rm_str = float(np.mean(lstr.isin(rstr_set))) if len(lstr) else np.nan

                    ev = np.nanmax([uc_raw, uc_str, rm_raw, rm_str])
                    ev = 0.0 if np.isnan(ev) else float(ev)

                    comp = 0.25 * sim + 0.65 * ev + 0.10 * (1.0 - lp["null_rate"])
                    comp = float(max(0.0, min(1.0, comp)))

                    scored.append({
                        "left_table": lt,
                        "right_table": rt,
                        "left_col": lc,
                        "right_col": rc,
                        "cardinality": int(rp["n_unique"]),   # right-side distinct count (generic key-likeness signal)
                        "name_similarity": float(sim),
                        "left_dtype": lp["dtype"],
                        "right_dtype": rp["dtype"],
                        "left_kind": lp["kind"],
                        "right_kind": rp["kind"],
                        "kind_compatible": bool(kind_compatible),
                        "unique_coverage_raw": uc_raw,
                        "unique_coverage_str": uc_str,
                        "row_match_rate_raw": rm_raw,
                        "row_match_rate_str": rm_str,
                        "evidence_max": ev,
                        "composite_score": comp,
                    })

            if scored:
                df_scored = pd.DataFrame(scored).sort_values("composite_score", ascending=False).head(top_k_per_pair)
                rows.append(df_scored)

    if not rows:
        return pd.DataFrame()

    out = pd.concat(rows, ignore_index=True)
    out = out.sort_values(["composite_score", "evidence_max", "name_similarity"], ascending=False).reset_index(drop=True)
    return out


In [7]:
# helper functions

def write_excel_no_sci(df, path, sheet_name, float_format="0.######"):
    """
    note: Writes a DataFrame to Excel and forces numeric columns to use a fixed
    decimal format (no scientific notation). Replaces the target sheet when it exists.
    Uses a safe mkdir pattern when the path has no directory component.
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    file_exists = os.path.exists(path)
    mode = "a" if file_exists else "w"
    writer_kwargs = dict(engine="openpyxl", mode=mode)
    if file_exists:
        writer_kwargs["if_sheet_exists"] = "replace"

    with pd.ExcelWriter(path, **writer_kwargs) as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        ws = writer.sheets[sheet_name]

        for idx, col in enumerate(df.columns, start=1):
            if pd.api.types.is_numeric_dtype(df[col]):
                col_letter = get_column_letter(idx)
                for cell in ws[col_letter]:
                    cell.number_format = float_format

def write_excel_blocks_one_sheet(
    blocks: list[tuple[str, pd.DataFrame]],
    path: str,
    sheet_name: str,
    float_format: str = "0.######",
    blank_rows_between: int = 2,
    replace_sheet: bool = True,
) -> None:
    """
    Write multiple (title, DataFrame) blocks into a SINGLE Excel worksheet.

    This exists because write_excel_no_sci() always replaces an entire sheet.
    Here, we intentionally write once per eval and stack multiple tables
    vertically within the same worksheet, preserving order and context.

    blocks:
        List of (title, DataFrame) tuples written top-to-bottom.
        Empty or None DataFrames are skipped.

    replace_sheet:
        If True, deletes and recreates the target sheet before writing.
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    if os.path.exists(path):
        wb = load_workbook(path)
    else:
        from openpyxl import Workbook
        wb = Workbook()
        if wb.sheetnames:
            wb.remove(wb[wb.sheetnames[0]])

    if sheet_name in wb.sheetnames and replace_sheet:
        wb.remove(wb[sheet_name])

    ws = wb.create_sheet(title=sheet_name)

    cur_row = 1

    for title, df in blocks:
        if df is None:
            continue
        if isinstance(df, pd.DataFrame) and df.empty:
            continue

        # Section title
        ws.cell(row=cur_row, column=1, value=str(title))
        cur_row += 1

        # DataFrame (header + rows)
        for r in dataframe_to_rows(df, index=False, header=True):
            ws.append(r)
            for c_idx, v in enumerate(r, start=1):
                if isinstance(v, (int, float, np.integer, np.floating)) and v is not None:
                    ws.cell(row=ws.max_row, column=c_idx).number_format = float_format

        # Spacer rows between blocks
        cur_row = ws.max_row + blank_rows_between

    wb.save(path)

def _sample_series(s: pd.Series, n: int, random_state: int) -> pd.Series:
    """
    note: Samples up to n non-null values from a Series to keep diagnostics fast on large tables.
    """
    ss = s.dropna()
    if len(ss) > n:
        return ss.sample(n=n, random_state=random_state)
    return ss

def _is_constant_like(n_unique: int, n_nonnull: int, const_unique_max: int = 1) -> bool:
    """
    note: Flags columns with ~1 unique value (or empty) as non-key candidates.
    """
    if n_nonnull <= 0:
        return True
    return int(n_unique) <= int(const_unique_max)

def _is_surrogate_index_like(
    s: pd.Series,
    sample_n: int = 50000,
    random_state: int = 7,
    allow_start_values: tuple[int, ...] = (0, 1),
    max_gap_frac: float = 0.01,
) -> bool:
    """
    note: Detects row-index-like columns (e.g., 0..n-1) using sample contiguity and low duplicates.
    """
    if not pd.api.types.is_numeric_dtype(s.dtype):
        return False
    ss = _sample_series(pd.to_numeric(s, errors="coerce"), n=sample_n, random_state=random_state).dropna()
    if len(ss) < 100:
        return False

    frac_intlike = float(np.mean(np.isclose(ss.to_numpy(), np.round(ss.to_numpy()))))
    if frac_intlike < 0.995:
        return False

    vals = np.sort(ss.astype(np.int64).to_numpy())
    vmin, vmax = int(vals[0]), int(vals[-1])
    if vmin not in allow_start_values:
        return False

    dup_rate = 1.0 - (len(np.unique(vals)) / len(vals))
    if dup_rate > 0.001:
        return False

    expected = (vmax - vmin + 1)
    unique = len(np.unique(vals))
    if expected <= 0:
        return False

    gap_frac = 1.0 - (unique / expected)
    return gap_frac <= float(max_gap_frac)


def _is_sentinel_date_like(
    s: pd.Series,
    sample_n: int = 50000,
    random_state: int = 7,
    sentinel_year_min: int = 2090,
    dominant_share_min: float = 0.90,
) -> bool:
    """
    note: Flags date-like columns with far-future sentinel values or near-constant date values
    (common in SCD metadata). Uses mixed-format parsing to avoid format-inference warnings.
    """
    ss = _sample_series(s.astype("string"), n=sample_n, random_state=random_state)
    if len(ss) < 200:
        return False

    # key change: avoid infer_datetime_format; use format="mixed" to reduce warnings
    dt = pd.to_datetime(ss, errors="coerce", format="mixed").dropna()
    if len(dt) < 50:
        return False

    if int(dt.dt.year.max()) >= int(sentinel_year_min):
        return True

    vc = dt.value_counts(dropna=True)
    if len(vc) == 0:
        return False
    return float(vc.iloc[0] / vc.sum()) >= float(dominant_share_min)

def _equivalence_groups_within_table(
    df: pd.DataFrame,
    sample_n: int = 50000,
    random_state: int = 7,
    min_eq_rate: float = 0.995,
    max_cols: int = 80,
) -> Tuple[dict, pd.DataFrame]:
    """
    note: Detects near-duplicate columns within a table using sampled row equality rates.
    """
    cols = list(df.columns)[:max_cols]
    if not cols:
        return {}, pd.DataFrame(columns=["col_a", "col_b", "eq_rate"])

    d = df[cols].copy()
    if len(d) > sample_n:
        d = d.sample(n=sample_n, random_state=random_state)

    norm = {}
    for c in cols:
        s = d[c]
        if pd.api.types.is_numeric_dtype(s.dtype):
            norm[c] = pd.to_numeric(s, errors="coerce")
        else:
            norm[c] = s.astype("string").fillna("")
    nd = pd.DataFrame(norm)

    parent = {c: c for c in cols}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    pairs = []
    for i in range(len(cols)):
        a = cols[i]
        sa = nd[a]
        for j in range(i + 1, len(cols)):
            b = cols[j]
            sb = nd[b]
            eq_rate = float((sa == sb).mean()) if len(sa) else np.nan
            if pd.notna(eq_rate) and eq_rate >= float(min_eq_rate):
                union(a, b)
                pairs.append((a, b, eq_rate))

    groups = {}
    for c in cols:
        groups.setdefault(find(c), []).append(c)

    col_to_gid = {}
    gid = 1
    for _, members in groups.items():
        if len(members) <= 1:
            continue
        for c in members:
            col_to_gid[c] = gid
        gid += 1

    return col_to_gid, pd.DataFrame(pairs, columns=["col_a", "col_b", "eq_rate"])

def _numeric_overlap_clusters(
    df: pd.DataFrame,
    sample_n: int = 50000,
    random_state: int = 7,
    min_support: int = 200,
    jaccard_min: float = 0.60,
    max_numeric_cols: int = 60,
) -> dict:
    """
    note: Groups numeric columns that frequently co-occur as non-zero within rows.
    Generic signal for overlapping classification fields (non-mutually-exclusive categories).
    """
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c].dtype)]
    num_cols = num_cols[:max_numeric_cols]
    if len(num_cols) < 2:
        return {}

    d = df[num_cols].copy()
    if len(d) > sample_n:
        d = d.sample(n=sample_n, random_state=random_state)

    m = (pd.DataFrame({c: pd.to_numeric(d[c], errors="coerce").fillna(0) for c in num_cols}) != 0)
    support = {c: int(m[c].sum()) for c in num_cols}

    parent = {c: c for c in num_cols}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(len(num_cols)):
        a = num_cols[i]
        if support[a] < min_support:
            continue
        for j in range(i + 1, len(num_cols)):
            b = num_cols[j]
            if support[b] < min_support:
                continue
            inter = int((m[a] & m[b]).sum())
            uni = int((m[a] | m[b]).sum())
            if uni <= 0:
                continue
            if (inter / uni) >= float(jaccard_min):
                union(a, b)

    clusters = {}
    for c in num_cols:
        clusters.setdefault(find(c), []).append(c)

    col_to_cluster = {}
    cid = 1
    for _, members in clusters.items():
        if len(members) <= 1:
            continue
        for c in members:
            col_to_cluster[c] = cid
        cid += 1

    return col_to_cluster

def _col_profile_table(
    df: pd.DataFrame,
    table_name: str,
    sample_n: int = 50000,
    random_state: int = 7,
) -> pd.DataFrame:
    """
    note: Builds per-column diagnostics inferred from distributions (nulls, uniqueness, sentinel patterns).
    """
    rows = []
    n_rows = int(len(df))
    for c in df.columns:
        s = df[c]
        nn = int(s.notna().sum())
        nun = int(s.nunique(dropna=True))
        null_rate = 1.0 - (nn / n_rows) if n_rows else 1.0
        unique_ratio = (nun / nn) if nn else np.nan

        rows.append({
            "table": table_name,
            "column": c,
            "dtype": str(s.dtype),
            "n_rows": n_rows,
            "n_nonnull": nn,
            "n_unique": nun,
            "null_rate": float(null_rate),
            "unique_ratio": float(unique_ratio) if pd.notna(unique_ratio) else np.nan,
            "constant_like": bool(_is_constant_like(nun, nn)),
            "surrogate_index_like": bool(_is_surrogate_index_like(s, sample_n=sample_n, random_state=random_state)),
            "sentinel_date_like": bool(_is_sentinel_date_like(s, sample_n=sample_n, random_state=random_state)),
        })
    return pd.DataFrame(rows)

def enrich_merge_audit(
    dfs: dict,
    recon: pd.DataFrame,
    sample_n: int = 50000,
    random_state: int = 7,
    sample_merge_rows: int = 20000,
) -> tuple[pd.DataFrame, dict]:
    """
    note: Enriches merge_audit with generalized diagnostics (no column-name logic).

    Adds:
    - left/right: constant_like, surrogate_index_like, sentinel_date_like
    - left/right: redundant_group_id (within-table equivalence clusters)
    - left/right: overlap_cluster_id (numeric non-zero co-occurrence clusters)
    - join-level: left/right unique_ratio + dup_rate
    - join-level: sample_merge_inflation_est (row inflation risk on a sampled left merge)
    """
    if recon is None or recon.empty:
        return recon, {}

    profiles = []
    redundant_pairs_blocks = []
    overlap_blocks = []
    col_to_gid_by_table = {}
    col_to_cluster_by_table = {}

    for t, df in dfs.items():
        prof = _col_profile_table(df, t, sample_n=sample_n, random_state=random_state)
        profiles.append(prof)

        gid_map, pairs_df = _equivalence_groups_within_table(df, sample_n=sample_n, random_state=random_state)
        col_to_gid_by_table[t] = gid_map
        if not pairs_df.empty:
            pairs_df = pairs_df.copy()
            pairs_df.insert(0, "table", t)
            redundant_pairs_blocks.append(pairs_df)

        cluster_map = _numeric_overlap_clusters(df, sample_n=sample_n, random_state=random_state)
        col_to_cluster_by_table[t] = cluster_map
        if cluster_map:
            ov = pd.DataFrame({
                "table": t,
                "column": list(cluster_map.keys()),
                "overlap_cluster_id": list(cluster_map.values()),
            }).sort_values(["table", "overlap_cluster_id", "column"])
            overlap_blocks.append(ov)

    prof_all = pd.concat(profiles, ignore_index=True) if profiles else pd.DataFrame()
    pairs_all = pd.concat(redundant_pairs_blocks, ignore_index=True) if redundant_pairs_blocks else pd.DataFrame(
        columns=["table", "col_a", "col_b", "eq_rate"]
    )
    overlap_all = pd.concat(overlap_blocks, ignore_index=True) if overlap_blocks else pd.DataFrame(
        columns=["table", "column", "overlap_cluster_id"]
    )

    prof_key = {(r["table"], r["column"]): r for _, r in prof_all.iterrows()} if not prof_all.empty else {}

    def _get_prof(table: str, col: str) -> dict:
        return prof_key.get((table, col), {})

    def _unique_ratio(table: str, col: str) -> float:
        return float(_get_prof(table, col).get("unique_ratio", np.nan))

    def _dup_rate(table: str, col: str) -> float:
        p = _get_prof(table, col)
        nn = p.get("n_nonnull", np.nan)
        nun = p.get("n_unique", np.nan)
        if pd.isna(nn) or pd.isna(nun) or nn == 0:
            return np.nan
        return float(max(0.0, (nn - nun) / nn))

    def _sample_merge_inflation(lt: str, rt: str, lc: str, rc: str) -> float:
        try:
            left = dfs[lt][[lc]].copy()
            right = dfs[rt][[rc]].copy()
            take = min(sample_merge_rows, len(left))
            if take <= 0:
                return np.nan
            left_s = left.sample(n=take, random_state=random_state)
            before = len(left_s)
            after = len(left_s.merge(right, how="left", left_on=lc, right_on=rc))
            return float((after / before) - 1.0) if before else np.nan
        except Exception:
            return np.nan

    out = recon.copy()

    for side in ("left", "right"):
        out[f"{side}_constant_like"] = out.apply(
            lambda r: bool(_get_prof(r[f"{side}_table"], r[f"{side}_col"]).get("constant_like", False)),
            axis=1
        )
        out[f"{side}_surrogate_index_like"] = out.apply(
            lambda r: bool(_get_prof(r[f"{side}_table"], r[f"{side}_col"]).get("surrogate_index_like", False)),
            axis=1
        )
        out[f"{side}_sentinel_date_like"] = out.apply(
            lambda r: bool(_get_prof(r[f"{side}_table"], r[f"{side}_col"]).get("sentinel_date_like", False)),
            axis=1
        )
        out[f"{side}_redundant_group_id"] = out.apply(
            lambda r: col_to_gid_by_table.get(r[f"{side}_table"], {}).get(r[f"{side}_col"], np.nan),
            axis=1
        )
        out[f"{side}_overlap_cluster_id"] = out.apply(
            lambda r: col_to_cluster_by_table.get(r[f"{side}_table"], {}).get(r[f"{side}_col"], np.nan),
            axis=1
        )

    out["right_unique_ratio"] = out.apply(lambda r: _unique_ratio(r["right_table"], r["right_col"]), axis=1)
    out["right_dup_rate"] = out.apply(lambda r: _dup_rate(r["right_table"], r["right_col"]), axis=1)
    out["left_unique_ratio"] = out.apply(lambda r: _unique_ratio(r["left_table"], r["left_col"]), axis=1)
    out["left_dup_rate"] = out.apply(lambda r: _dup_rate(r["left_table"], r["left_col"]), axis=1)
    out["sample_merge_inflation_est"] = out.apply(
        lambda r: _sample_merge_inflation(r["left_table"], r["right_table"], r["left_col"], r["right_col"]),
        axis=1
    )

    table_diags = {
        "column_profiles": prof_all,
        "redundant_pairs": pairs_all,
        "overlap_clusters": overlap_all,
    }
    return out, table_diags

def add_join_cardinality_labels(
    recon_df: pd.DataFrame,
    *,
    keylike_unique_ratio: float = 0.98,
    explosion_medium: float = 0.001,
    explosion_high: float = 0.01,
) -> pd.DataFrame:
    """
    note: Adds join cardinality labels (1:1, 1:M, M:1, M:M) and an English-readable
    summary column to a recon/enriched recon DataFrame. Uses only generic metrics:
    unique ratios and sampled merge inflation. No table/column name logic.
    """
    if recon_df is None or recon_df.empty:
        return recon_df

    out = recon_df.copy()

    def is_keylike(u):
        if pd.isna(u):
            return None
        return bool(u >= keylike_unique_ratio)

    labels = []
    english = []

    for _, r in out.iterrows():
        lk = is_keylike(r.get("left_unique_ratio", np.nan))
        rk = is_keylike(r.get("right_unique_ratio", np.nan))
        infl = r.get("sample_merge_inflation_est", np.nan)

        # inflation override: if it empirically explodes, treat as M:M risk
        if pd.notna(infl) and infl >= explosion_high:
            lab = "M:M"
            msg = "High join explosion risk (sample inflation high)."
        else:
            if lk is None or rk is None:
                lab = "unknown"
                msg = "Insufficient uniqueness metrics to classify."
            elif lk and rk:
                lab = "1:1"
                msg = "Both sides look key-like (near-unique)."
            elif lk and (not rk):
                lab = "1:M"
                msg = "Left looks key-like; right has duplicates (right not unique)."
            elif (not lk) and rk:
                lab = "M:1"
                msg = "Right looks key-like; left has duplicates (left not unique)."
            else:
                lab = "M:M"
                msg = "Both sides have duplicates (neither side unique)."

            # medium inflation hint (keep label, but add caution)
            if pd.notna(infl) and infl >= explosion_medium and lab != "unknown":
                msg = msg + " Sample inflation suggests possible row growth."

        labels.append(lab)
        english.append(msg)

    out["join_cardinality_label"] = labels
    out["join_cardinality_explainer"] = english
    return out

def left_join_coverage(base_df, right_df, base_col, join_col, sample_rows=200000, random_state=7):
    """
    Estimates left-join coverage and inner-join loss using generic set membership.
    Returns rates on a row sample for speed.
    """
    s = base_df[[base_col]].dropna()
    if len(s) > sample_rows:
        s = s.sample(n=sample_rows, random_state=random_state)

    right_keys = set(right_df[join_col].dropna().unique().tolist())

    match_rate = float(s[base_col].isin(right_keys).mean()) if len(s) else np.nan
    null_after_left = 1.0 - match_rate if pd.notna(match_rate) else np.nan

    return {
        "base_col": base_col,
        "join_col": join_col,
        "sample_rows": int(len(s)),
        "match_rate": match_rate,
        "null_after_left_join": null_after_left,
        "inner_join_row_loss_rate_est": null_after_left,
    }

def add_recon_reason_flags(
    recon_enriched: pd.DataFrame,
    *,
    unique_ratio_warn: float = 0.98,
    dup_rate_warn: float = 0.01,
    inflation_warn: float = 0.001,
    inflation_fail: float = 0.01,
) -> pd.DataFrame:
    """
    note: Adds generalized, non-domain reason flags to merge_audit rows to support
    human review and safer automation. No table/column naming logic is used.
    Flags are derived from enriched diagnostics columns such as constant_like,
    surrogate_index_like, sentinel_date_like, redundancy group ids, overlap clusters,
    uniqueness/dup rates, and sampled merge inflation.
    """
    if recon_enriched is None or recon_enriched.empty:
        return recon_enriched

    out = recon_enriched.copy()

    def _bucket_inflation(x: float) -> str:
        """note: Buckets sampled merge inflation into low/medium/high for row explosion risk triage."""
        if pd.isna(x):
            return "unknown"
        if x >= inflation_fail:
            return "high"
        if x >= inflation_warn:
            return "medium"
        return "low"

    def _bucket_key_quality(unique_ratio: float, dup_rate: float) -> str:
        """note: Buckets key quality using uniqueness ratio and duplicate rate on the join side."""
        if pd.isna(unique_ratio) and pd.isna(dup_rate):
            return "unknown"
        # treat unknowns conservatively
        ur = unique_ratio if pd.notna(unique_ratio) else 0.0
        dr = dup_rate if pd.notna(dup_rate) else 1.0
        if ur < unique_ratio_warn or dr > dup_rate_warn:
            return "risky"
        return "ok"

    flags = []
    for _, r in out.iterrows():
        row_flags = []

        # key-quality / non-key heuristics (either side)
        if bool(r.get("left_constant_like", False)) or bool(r.get("right_constant_like", False)):
            row_flags.append("constant_like_column")
        if bool(r.get("left_surrogate_index_like", False)) or bool(r.get("right_surrogate_index_like", False)):
            row_flags.append("surrogate_index_like_column")
        if bool(r.get("left_sentinel_date_like", False)) or bool(r.get("right_sentinel_date_like", False)):
            row_flags.append("sentinel_date_like_column")

        # redundancy within table (either side)
        if pd.notna(r.get("left_redundant_group_id", np.nan)) or pd.notna(r.get("right_redundant_group_id", np.nan)):
            row_flags.append("within_table_redundant_column_group")

        # numeric overlap clusters (either side) indicates “classification-like” overlapping fields
        if pd.notna(r.get("left_overlap_cluster_id", np.nan)) or pd.notna(r.get("right_overlap_cluster_id", np.nan)):
            row_flags.append("numeric_overlap_cluster_member")

        # right-side join safety (non-unique join key risk)
        ru = r.get("right_unique_ratio", np.nan)
        rd = r.get("right_dup_rate", np.nan)
        if _bucket_key_quality(ru, rd) == "risky":
            row_flags.append("right_join_key_not_unique_like")

        # sampled row inflation (proxy for many-to-many explosion)
        infl = r.get("sample_merge_inflation_est", np.nan)
        infl_bucket = _bucket_inflation(infl)
        if infl_bucket in {"medium", "high"}:
            row_flags.append(f"sample_merge_inflation_{infl_bucket}")

        flags.append(";".join(row_flags) if row_flags else "")

    out["reason_flags"] = flags

    # optional: compact severity fields for sorting/filtering
    out["inflation_risk_bucket"] = out["sample_merge_inflation_est"].apply(_bucket_inflation)
    out["right_key_quality_bucket"] = [
        _bucket_key_quality(u, d) for u, d in zip(out["right_unique_ratio"], out["right_dup_rate"])
    ]

    return out

# join audits

def post_join_audit(
    base_df: pd.DataFrame,
    joined_df: pd.DataFrame,
    top_null_columns: int = 25,
) -> dict:
    """
    Minimal post-join audit that stays generic and low-noise.

    Outputs:
    - row_integrity: confirms no row explosion and summarizes column growth
    - new_column_null_coverage: null rates for newly added columns only
    - new_column_null_coverage_top: top-N null-rate columns for quick review

    Does not assume any table names and does not mutate inputs.
    """
    out = {}

    base_rows = len(base_df)
    joined_rows = len(joined_df)

    out["row_integrity"] = pd.DataFrame([{
        "base_rows": base_rows,
        "joined_rows": joined_rows,
        "delta_rows": joined_rows - base_rows,
        "pct_change": (joined_rows / base_rows - 1.0) if base_rows else np.nan,
        "base_cols": base_df.shape[1],
        "joined_cols": joined_df.shape[1],
        "added_cols": joined_df.shape[1] - base_df.shape[1],
    }])

    base_cols = list(base_df.columns)
    new_cols = [c for c in joined_df.columns if c not in base_cols]

    if new_cols:
        null_rates = joined_df[new_cols].isna().mean().sort_values(ascending=False)
        cov = pd.DataFrame({
            "column": null_rates.index,
            "null_rate": null_rates.values,
            "non_null_rate": 1.0 - null_rates.values,
        })
        out["new_column_null_coverage"] = cov
        out["new_column_null_coverage_top"] = cov.head(top_null_columns)
    else:
        out["new_column_null_coverage"] = pd.DataFrame(columns=["column", "null_rate", "non_null_rate"])
        out["new_column_null_coverage_top"] = out["new_column_null_coverage"].copy()

    return out

def stack_audit_tables(audit: dict) -> pd.DataFrame:
    """
    Stacks all audit DataFrames into one table with an audit_section label.
    Skips non-DataFrame or empty items.
    """
    blocks = []

    for name, df in audit.items():
        if not isinstance(df, pd.DataFrame) or df.empty:
            continue

        block = df.copy()
        block.insert(0, "audit_section", name)
        blocks.append(block)

    return pd.concat(blocks, ignore_index=True, sort=False) if blocks else pd.DataFrame()

def write_dataframe_at(
    ws,
    df: pd.DataFrame,
    start_row: int,
    start_col: int,
    *,
    float_format: str = "0.######",
    include_header: bool = True,
) -> tuple[int, int]:
    """
    note: Writes a DataFrame into an existing openpyxl worksheet at (start_row, start_col).
    Applies number_format to numeric cells. Returns (end_row, end_col) occupied by the write.
    """
    if df is None or (isinstance(df, pd.DataFrame) and df.empty):
        return start_row - 1, start_col - 1

    cur_row = start_row

    rows = dataframe_to_rows(df, index=False, header=include_header)
    max_width = 0

    for r in rows:
        max_width = max(max_width, len(r))
        for j, v in enumerate(r, start=0):
            cell = ws.cell(row=cur_row, column=start_col + j, value=v)
            if isinstance(v, (int, float, np.integer, np.floating)) and v is not None:
                cell.number_format = float_format
        cur_row += 1

    end_row = cur_row - 1
    end_col = start_col + max_width - 1
    return end_row, end_col

def write_excel_blocks_left_to_right_one_sheet(
    blocks: list[tuple[str, pd.DataFrame]],
    path: str,
    sheet_name: str,
    *,
    float_format: str = "0.######",
    blank_cols_between: int = 2,
    start_row: int = 1,
    start_col: int = 1,
    replace_sheet: bool = True,
) -> tuple[int, int]:
    """
    note: Writes multiple (title, DataFrame) blocks left-to-right across a single worksheet.
    Returns (max_end_row, max_end_col) used by any block.
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    if os.path.exists(path):
        wb = load_workbook(path)
    else:
        wb = Workbook()

    if sheet_name in wb.sheetnames and replace_sheet:
        wb.remove(wb[sheet_name])

    ws = wb.create_sheet(title=sheet_name) if sheet_name not in wb.sheetnames else wb[sheet_name]

    cur_col = start_col
    max_end_row = start_row - 1
    max_end_col = start_col - 1

    for title, df in blocks:
        if df is None:
            continue
        if isinstance(df, pd.DataFrame) and df.empty:
            continue

        ws.cell(row=start_row, column=cur_col, value=str(title))

        end_row, end_col = write_dataframe_at(
            ws,
            df,
            start_row=start_row + 1,
            start_col=cur_col,
            float_format=float_format,
            include_header=True,
        )

        max_end_row = max(max_end_row, end_row)
        max_end_col = max(max_end_col, end_col)

        cur_col = end_col + blank_cols_between + 1

    wb.save(path)
    return max_end_row, max_end_col

def _detect_join_key_cols(df: pd.DataFrame, *, side: str) -> tuple[str, str]:
    """
    note: Detects the (table_col, field_col) pair used to identify joins in either a plan table
    or a recon table. side must be "left" or "right". Raises ValueError if not found.
    """
    candidates = []
    if side == "left":
        candidates = [("base_table", "base_col"), ("left_table", "left_col")]
    elif side == "right":
        candidates = [("join_table", "join_col"), ("right_table", "right_col")]
    else:
        raise ValueError("side must be 'left' or 'right'.")

    cols = set(df.columns)
    for tcol, ccol in candidates:
        if tcol in cols and ccol in cols:
            return tcol, ccol

    raise ValueError(f"Could not detect join key columns for side={side} in df columns={list(df.columns)}")

def build_plan_recon_review(
    plan_df: pd.DataFrame,
    recon_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    note: Builds a recon review table filtered to only the joins present in the plan.
    Matches on (left_table,left_col,right_table,right_col) equivalents between plan and recon.
    Outputs a reduced, English-readable set of recon columns and drops fully-empty rows.
    """
    if plan_df is None or plan_df.empty or recon_df is None or recon_df.empty:
        return pd.DataFrame()

    p_lt, p_lc = _detect_join_key_cols(plan_df, side="left")
    p_rt, p_rc = _detect_join_key_cols(plan_df, side="right")

    r_lt, r_lc = _detect_join_key_cols(recon_df, side="left")
    r_rt, r_rc = _detect_join_key_cols(recon_df, side="right")

    pkeys = plan_df[[p_lt, p_lc, p_rt, p_rc]].dropna().drop_duplicates().copy()
    pkeys.columns = ["_lt", "_lc", "_rt", "_rc"]

    r = recon_df.copy()
    r["_lt"] = r[r_lt]
    r["_lc"] = r[r_lc]
    r["_rt"] = r[r_rt]
    r["_rc"] = r[r_rc]

    keep_cols = [
        r_lt, r_lc, r_rt, r_rc,
        "join_cardinality_label",
        "inflation_risk_bucket",
        "right_key_quality_bucket",
        "name_similarity",
        "sample_merge_inflation_est",
        "left_unique_ratio", "right_unique_ratio",
        "left_dup_rate", "right_dup_rate",
        "reason_flags",
        "join_cardinality_explainer",
    ]
    keep_cols = [c for c in keep_cols if c in r.columns]

    merged = pkeys.merge(
        r[["_lt", "_lc", "_rt", "_rc"] + keep_cols],
        on=["_lt", "_lc", "_rt", "_rc"],
        how="left",
    )

    merged = merged.drop(columns=["_lt", "_lc", "_rt", "_rc"], errors="ignore")

    # Drop rows that are visually “blank” (all NaN / empty-string after merge)
    if not merged.empty:
        key_cols = [c for c in [r_lt, r_lc, r_rt, r_rc] if c in merged.columns]
        if key_cols:
            merged = merged.dropna(subset=key_cols, how="all")

        # Extra guard: remove rows where every cell is NaN/empty-string
        def _row_has_any_value(row) -> bool:
            for v in row.values:
                if v is None:
                    continue
                if isinstance(v, float) and pd.isna(v):
                    continue
                if isinstance(v, str) and v.strip() == "":
                    continue
                return True
            return False

        merged = merged[merged.apply(_row_has_any_value, axis=1)].reset_index(drop=True)

    return merged

def write_merge_audit_sheet(
    path: str,
    recon_df: pd.DataFrame,
    diags: dict,
    *,
    audit_blocks: list[tuple[str, pd.DataFrame]] | None = None,
    sheet_name: str = "merge_audit",
    float_format: str = "0.######",
    blank_cols_between: int = 2,
    blank_rows_between_sections: int = 2,
) -> None:
    """
    note: Writes a single-sheet merge_audit layout:
    - diagnostics blocks + optional audit blocks left-to-right at the top
    - main merge_audit table below those blocks
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    if os.path.exists(path):
        wb = load_workbook(path)
    else:
        wb = Workbook()

    if sheet_name in wb.sheetnames:
        wb.remove(wb[sheet_name])
    ws = wb.create_sheet(title=sheet_name)

    blocks = [
        ("column_profiles", diags.get("column_profiles", pd.DataFrame())),
        ("redundant_pairs", diags.get("redundant_pairs", pd.DataFrame())),
        ("overlap_clusters", diags.get("overlap_clusters", pd.DataFrame())),
    ]
    if audit_blocks:
        blocks.extend([(t, d) for (t, d) in audit_blocks if isinstance(d, pd.DataFrame) and not d.empty])

    cur_col = 1
    start_row = 1
    max_end_row = 0

    for title, df in blocks:
        if df is None or (isinstance(df, pd.DataFrame) and df.empty):
            continue

        ws.cell(row=start_row, column=cur_col, value=str(title))
        end_row, end_col = write_dataframe_at(
            ws,
            df,
            start_row=start_row + 1,
            start_col=cur_col,
            float_format=float_format,
            include_header=True,
        )
        max_end_row = max(max_end_row, end_row)
        cur_col = end_col + blank_cols_between + 1

    table_start_row = (max_end_row + blank_rows_between_sections + 1) if max_end_row > 0 else 1
    ws.cell(row=table_start_row, column=1, value="merge_audit")
    write_dataframe_at(
        ws,
        recon_df,
        start_row=table_start_row + 1,
        start_col=1,
        float_format=float_format,
        include_header=True,
    )

    wb.save(path)

def write_merge_plan_sheet(
    path: str,
    plan_df: pd.DataFrame,
    plan_recon_review_df: pd.DataFrame,
    *,
    sheet_name: str = "merge_plan",
    float_format: str = "0.######",
    blank_rows_between_sections: int = 2,
) -> None:
    """
    note: Writes a single-sheet merge_plan layout:
    - merge_plan table at the top (with projected_new_col_examples omitted if present)
    - plan-only recon review table below it
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    if os.path.exists(path):
        wb = load_workbook(path)
    else:
        wb = Workbook()

    if sheet_name in wb.sheetnames:
        wb.remove(wb[sheet_name])
    ws = wb.create_sheet(title=sheet_name)

    # omit the requested column at write-time
    plan_out = plan_df.copy() if plan_df is not None else pd.DataFrame()
    if "projected_new_col_examples" in plan_out.columns:
        plan_out = plan_out.drop(columns=["projected_new_col_examples"])

    ws.cell(row=1, column=1, value="merge_plan")
    end_row, _end_col = write_dataframe_at(
        ws,
        plan_out,
        start_row=2,
        start_col=1,
        float_format=float_format,
        include_header=True,
    )

    plan_recon_review_out = plan_recon_review_df.copy() if plan_recon_review_df is not None else pd.DataFrame()
    # avoid visually blank rows
    if not plan_recon_review_out.empty:
        plan_recon_review_out = plan_recon_review_out.dropna(how="all")

    review_start = (end_row + blank_rows_between_sections + 1) if end_row >= 2 else 1
    ws.cell(row=review_start, column=1, value="plan_recon_review")
    write_dataframe_at(
        ws,
        plan_recon_review_out,
        start_row=review_start + 1,
        start_col=1,
        float_format=float_format,
        include_header=True,
    )

    wb.save(path)


# human in the loop - automated join

In [8]:
# automated join funcs

def build_merge_plan(
    dfs: dict,
    recon: pd.DataFrame,
    base_table: str | None = None,
    min_evidence: float = 0.95,
    min_name_similarity: float = 0.70,
    min_right_unique_ratio: float = 0.98,
    allow_direction_flip: bool = True,
    sample_merge_rows: int = 50000,
    explode_tol_warn: float = 0.001,
    explode_tol_fail: float = 0.01,
    coverage_warn: float = 0.05,
    coverage_fail: float = 0.20,
    random_state: int = 7,
) -> pd.DataFrame:
    """
    Builds a conservative join plan from a join-recon DataFrame and adds
    generic, pre-execution risk flags for:
    - row explosion risk (many-to-many / duplicated right keys)
    - coverage risk (expected unmatched rows / NaNs after left join)

    No domain assumptions: no special casing of names, SCD fields, or fact/dim roles.
    """

    rng = np.random.default_rng(random_state)

    if base_table is None:
        base_table = max(dfs.keys(), key=lambda k: len(dfs[k]))

    required = {
        "left_table", "right_table", "left_col", "right_col",
        "evidence_max", "name_similarity", "composite_score"
    }
    missing = required - set(recon.columns)
    if missing:
        raise ValueError(f"recon is missing required columns: {sorted(missing)}")

    def col_stats(df: pd.DataFrame, col: str) -> dict:
        s = df[col]
        n = len(s)
        nn = int(s.notna().sum())
        nun = int(s.nunique(dropna=True))
        null_rate = 1.0 - (nn / n) if n else 1.0
        unique_ratio = (nun / nn) if nn else 0.0
        dup_nonnull = max(0, nn - nun)
        dup_rate = (dup_nonnull / nn) if nn else 0.0
        avg_matches_per_key = (nn / nun) if nun else np.inf
        return {
            "n_rows": int(n),
            "n_nonnull": nn,
            "n_unique": nun,
            "null_rate": float(null_rate),
            "unique_ratio": float(unique_ratio),
            "dup_nonnull": int(dup_nonnull),
            "dup_rate": float(dup_rate),
            "avg_matches_per_key": float(avg_matches_per_key),
            "dtype": str(s.dtype),
        }

    def best_match_rate(row: pd.Series) -> float:
        # prefer row_match_rate_* if present; otherwise fall back to evidence_max
        candidates = []
        for c in ("row_match_rate_str", "row_match_rate_raw"):
            if c in row.index and pd.notna(row[c]):
                candidates.append(float(row[c]))
        if candidates:
            return max(candidates)
        return float(row.get("evidence_max", np.nan))

    def sample_merge_inflation(
        base_df: pd.DataFrame,
        right_df: pd.DataFrame,
        base_col: str,
        join_col: str,
    ) -> float:
        # sample base rows only (cheap) and join on a skinny projection
        n = len(base_df)
        if n == 0:
            return 0.0

        take = min(sample_merge_rows, n)
        base_s = base_df[[base_col]].sample(n=take, random_state=random_state)
        right_s = right_df[[join_col]]

        before = len(base_s)
        after = len(base_s.merge(right_s, how="left", left_on=base_col, right_on=join_col))
        if before == 0:
            return 0.0
        return (after / before) - 1.0

    def risk_bucket(value: float, warn: float, fail: float) -> str:
        if pd.isna(value):
            return "unknown"
        if value >= fail:
            return "high"
        if value >= warn:
            return "medium"
        return "low"

    plan_rows = []
    other_tables = [t for t in dfs.keys() if t != base_table]

    for t in other_tables:
        c1 = recon[(recon["left_table"] == base_table) & (recon["right_table"] == t)].copy()
        c2 = recon[(recon["left_table"] == t) & (recon["right_table"] == base_table)].copy() if allow_direction_flip else pd.DataFrame()
        candidates = pd.concat([c1, c2], ignore_index=True)

        if candidates.empty:
            plan_rows.append({"base_table": base_table, "join_table": t, "status": "no_candidate"})
            continue

        best = candidates.sort_values(
            ["composite_score", "evidence_max", "name_similarity"],
            ascending=False
        ).iloc[0]

        # normalize direction so we always plan a base <- left join right_table
        if best["left_table"] == base_table:
            base_col = best["left_col"]
            join_col = best["right_col"]
        else:
            base_col = best["right_col"]
            join_col = best["left_col"]

        bs = col_stats(dfs[base_table], base_col)
        js = col_stats(dfs[t], join_col)

        expected_match_rate = best_match_rate(best)
        expected_null_after_left_join = 1.0 - expected_match_rate if pd.notna(expected_match_rate) else np.nan

        # sampled merge inflation (pre-join explosion proxy)
        try:
            inflation = sample_merge_inflation(dfs[base_table], dfs[t], base_col, join_col)
        except Exception:
            inflation = np.nan

        explode_risk = max(
            js["avg_matches_per_key"] - 1.0,  # duplicate-key pressure
            inflation if pd.notna(inflation) else 0.0,  # empirical pressure
        )
        explode_bucket = risk_bucket(explode_risk, explode_tol_warn, explode_tol_fail)
        coverage_bucket = risk_bucket(expected_null_after_left_join, coverage_warn, coverage_fail)

        passes = (
            float(best["evidence_max"]) >= min_evidence and
            float(best["name_similarity"]) >= min_name_similarity and
            js["unique_ratio"] >= min_right_unique_ratio and
            explode_bucket != "high"
        )

        flag_reasons = []
        if float(best["evidence_max"]) < min_evidence or float(best["name_similarity"]) < min_name_similarity:
            flag_reasons.append("low_evidence_or_name_similarity")
        if js["unique_ratio"] < min_right_unique_ratio:
            flag_reasons.append("right_key_not_unique")
        if explode_bucket == "high":
            flag_reasons.append("row_explosion_risk_high")
        if coverage_bucket in {"medium", "high"}:
            flag_reasons.append("coverage_risk_" + coverage_bucket)

        plan_rows.append({
            "base_table": base_table,
            "join_table": t,
            "base_col": base_col,
            "join_col": join_col,
            "join_how": "left",

            "composite_score": float(best["composite_score"]),
            "evidence_max": float(best["evidence_max"]),
            "name_similarity": float(best["name_similarity"]),

            "base_dtype": bs["dtype"],
            "join_dtype": js["dtype"],
            "base_null_rate": bs["null_rate"],
            "join_null_rate": js["null_rate"],

            "right_n_nonnull": js["n_nonnull"],
            "right_n_unique": js["n_unique"],
            "right_unique_ratio": js["unique_ratio"],
            "right_dup_nonnull": js["dup_nonnull"],
            "right_dup_rate": js["dup_rate"],
            "right_avg_matches_per_key": js["avg_matches_per_key"],

            "expected_match_rate": float(expected_match_rate) if pd.notna(expected_match_rate) else np.nan,
            "expected_null_after_left_join": float(expected_null_after_left_join) if pd.notna(expected_null_after_left_join) else np.nan,

            "sample_merge_inflation": float(inflation) if pd.notna(inflation) else np.nan,
            "explode_risk_score": float(explode_risk) if pd.notna(explode_risk) else np.nan,
            "explode_risk": explode_bucket,
            "coverage_risk": coverage_bucket,

            "status": "ok" if passes else "flag",
            "flag_reason": "" if passes else ";".join(flag_reasons),
        })

    plan = pd.DataFrame(plan_rows)

    status_order = {"ok": 0, "flag": 1, "no_candidate": 2}
    plan["__ord"] = plan["status"].map(status_order).fillna(9).astype(int)
    plan = plan.sort_values(["__ord", "explode_risk", "coverage_risk", "join_table"]).drop(columns="__ord").reset_index(drop=True)
    return plan

def execute_merge_plan(
    dfs: dict,
    plan: pd.DataFrame,
    explode_tolerance: float = 0.001,
    rename_strategy: str = "prefix",   # "prefix", "suffix", or "none"
    prefix_sep: str = "__",
    suffix: str = "_dim",
    keep_plan_only: bool = True,
) -> pd.DataFrame:
    """
    Executes a join plan conservatively with optional, generic column renaming.

    rename_strategy:
    - "prefix": right table's non-key columns become f"{join_table}__{col}"
    - "suffix": uses pandas merge suffixes (less informative provenance)
    - "none": no renaming (risk: collisions / ambiguity)

    Guardrails:
    - Skips non-ok joins unless keep_plan_only=False
    - Left joins onto base table
    - Fails closed on row explosion beyond explode_tolerance
    """
    if plan.empty:
        raise ValueError("plan is empty")

    base_table = plan.loc[0, "base_table"]
    out = dfs[base_table].copy()
    base_n = len(out)

    for _, r in plan.iterrows():
        if r.get("status") != "ok":
            if keep_plan_only:
                continue

        jt = r["join_table"]
        base_col = r["base_col"]
        join_col = r["join_col"]

        right = dfs[jt].copy()

        # refuse duplicated right keys (dimension must be key-unique for safe auto-join)
        if right[join_col].dropna().duplicated().any():
            if keep_plan_only:
                continue
            raise ValueError(f"Right key not unique: {jt}.{join_col}")

        # rename right columns for provenance and collision avoidance
        if rename_strategy == "prefix":
            ren = {}
            for c in right.columns:
                if c == join_col:
                    continue
                ren[c] = f"{jt}{prefix_sep}{c}"
            right = right.rename(columns=ren)

            # if we renamed non-key cols only, join_col name stays the same
            right_join_col = join_col

        elif rename_strategy == "suffix":
            right_join_col = join_col  # suffixing happens via merge suffixes

        elif rename_strategy == "none":
            right_join_col = join_col

        else:
            raise ValueError("rename_strategy must be one of: 'prefix', 'suffix', 'none'")

        before_n = len(out)

        if rename_strategy == "suffix":
            out = out.merge(
                right,
                how="left",
                left_on=base_col,
                right_on=right_join_col,
                suffixes=("", suffix),
            )
        else:
            out = out.merge(
                right,
                how="left",
                left_on=base_col,
                right_on=right_join_col,
            )

        after_n = len(out)

        if after_n > int(np.ceil(before_n * (1.0 + explode_tolerance))):
            raise ValueError(
                f"Join {base_table}.{base_col} -> {jt}.{join_col} increased rows "
                f"from {before_n} to {after_n}. Many-to-many risk."
            )

    if len(out) > int(np.ceil(base_n * (1.0 + explode_tolerance))):
        raise ValueError(f"Final joined table row count {len(out)} exceeds base {base_n} beyond tolerance.")

    return out

def add_plan_projection(
    dfs: dict,
    plan: pd.DataFrame,
    rename_strategy: str = "prefix",   # "prefix", "suffix", "none"
    prefix_sep: str = "__",
    suffix: str = "_dim",
) -> pd.DataFrame:
    """
    Adds a generic projection to a join plan so you can anticipate output width and
    column naming before executing joins.

    Adds:
    - base_n_cols
    - right_n_cols
    - projected_added_cols (per join)
    - projected_total_cols_after (cumulative)
    """
    if plan.empty:
        raise ValueError("plan is empty")
    if "base_table" not in plan.columns or "join_table" not in plan.columns or "join_col" not in plan.columns:
        raise ValueError("plan must contain base_table, join_table, join_col columns")

    base_table = plan.loc[0, "base_table"]
    base_cols = list(dfs[base_table].columns)
    base_n_cols = len(base_cols)

    total = base_n_cols
    out_rows = []

    for _, r in plan.iterrows():
        jt = r["join_table"]
        join_col = r.get("join_col", None)

        if jt not in dfs:
            out_rows.append({**r.to_dict(),
                "base_n_cols": base_n_cols,
                "right_n_cols": np.nan,
                "projected_added_cols": np.nan,
                "projected_total_cols_after": total,
            })
            continue

        right_cols = list(dfs[jt].columns)
        right_n_cols = len(right_cols)

        # added cols assumes we don't duplicate the join key column name in output
        # (merge will include both if key names differ; your plans generally match names)
        added = right_n_cols - (1 if join_col in right_cols else 0)

        # project resulting column names (examples)
        examples = []

        if rename_strategy == "prefix":
            for c in right_cols:
                if c == join_col:
                    continue
                examples.append(f"{jt}{prefix_sep}{c}")
        elif rename_strategy == "suffix":
            # only collisions would be suffixed; generic projection shows raw names as best effort
            # because predicting collisions requires checking base columns
            for c in right_cols:
                if c == join_col:
                    continue
                if c in base_cols:
                    examples.append(f"{c}{suffix}")
                else:
                    examples.append(c)
        elif rename_strategy == "none":
            for c in right_cols:
                if c == join_col:
                    continue
                examples.append(c)
        else:
            raise ValueError("rename_strategy must be one of: 'prefix', 'suffix', 'none'")

        total += max(0, added)

        out_rows.append({
            **r.to_dict(),
            "base_n_cols": base_n_cols,
            "right_n_cols": right_n_cols,
            "projected_added_cols": int(max(0, added)),
            "projected_total_cols_after": int(total),
        })

    return pd.DataFrame(out_rows)

def merge_plan_and_projection(plan: pd.DataFrame, plan_proj: pd.DataFrame) -> pd.DataFrame:
    """
    Merges join plan and projection outputs into a single table.

    Assumes both DataFrames:
    - have identical row order
    - represent the same join candidates

    Keeps plan columns first, then appends projection-only fields.
    """
    proj_only_cols = [
        c for c in plan_proj.columns
        if c not in plan.columns
    ]

    merged = pd.concat(
        [plan.reset_index(drop=True), plan_proj[proj_only_cols].reset_index(drop=True)],
        axis=1
    )
    return merged


In [9]:
# recon generation + write merge_audit sheet

recon_base = merge_audit(dfs, top_k_per_pair=5, name_sim_min=0.60)

recon_enriched, diags = enrich_merge_audit(
    dfs,
    recon_base,
    sample_n=50_000,
    random_state=7,
    sample_merge_rows=20_000,
)

recon_enriched = add_recon_reason_flags(recon_enriched)
recon_enriched = add_join_cardinality_labels(recon_enriched)

write_merge_audit_sheet(
    AUDIT_XLSX,
    recon_enriched,
    diags,
    audit_blocks=(audit_blocks if "audit_blocks" in globals() else None),
    sheet_name="merge_audit",
    float_format="0.######",
    blank_cols_between=2,
    blank_rows_between_sections=2,
)

recon_enriched.head(15)


,left_table,right_table,left_col,right_col,cardinality,name_similarity,left_dtype,right_dtype,left_kind,right_kind,kind_compatible,unique_coverage_raw,unique_coverage_str,row_match_rate_raw,row_match_rate_str,evidence_max,composite_score,left_constant_like,left_surrogate_index_like,left_sentinel_date_like,left_redundant_group_id,left_overlap_cluster_id,right_constant_like,right_surrogate_index_like,right_sentinel_date_like,right_redundant_group_id,right_overlap_cluster_id,right_unique_ratio,right_dup_rate,left_unique_ratio,left_dup_rate,sample_merge_inflation_est,reason_flags,inflation_risk_bucket,right_key_quality_bucket,join_cardinality_label,join_cardinality_explainer
0,GSCM 521 - Canvas Gradebook Data_clean,GSCM 521 - Canvas Interaction Data v2_clean,sis_id,sis_id,618,1.000000,object,object,stringlike,stringlike,True,0.998239,0.998239,0.998239,0.998239,0.998239,0.998856,False,False,False,NaN,NaN,False,False,False,NaN,NaN,0.002238,0.997762,1.000000,0.000000,469.556338,right_join_key_not_unique_like;sample_merge_in...,high,risky,M:M,High join explosion risk (sample inflation high).
1,GSCM 521 - Canvas Gradebook Data_clean,GSCM 521 - Banner Grade Data v2_clean,random_student,random_student,629,1.000000,object,object,stringlike,stringlike,True,0.978873,0.978873,0.978873,0.978873,0.978873,0.986268,False,False,False,NaN,NaN,False,False,False,NaN,NaN,0.714773,0.285227,1.000000,0.000000,0.429577,right_join_key_not_unique_like;sample_merge_in...,high,risky,M:M,High join explosion risk (sample inflation high).
2,GSCM 521 - Canvas Interaction Data v2_clean,GSCM 521 - Canvas Gradebook Data_clean,sis_id,sis_id,568,1.000000,object,object,stringlike,stringlike,True,0.917476,0.917476,0.968200,0.968200,0.968200,0.979330,False,False,False,NaN,NaN,False,False,False,NaN,NaN,1.000000,0.000000,0.002238,0.997762,0.000000,,low,ok,M:1,Right looks key-like; left has duplicates (lef...
3,GSCM 521 - Banner Grade Data v2_clean,GSCM 521 - Canvas Gradebook Data_clean,random_student,random_student,568,1.000000,object,object,stringlike,stringlike,True,0.883943,0.883943,0.909091,0.909091,0.909091,0.940909,False,False,False,NaN,NaN,False,False,False,NaN,NaN,1.000000,0.000000,0.714773,0.285227,0.000000,,low,ok,M:1,Right looks key-like; left has duplicates (lef...
4,GSCM 521 - Banner Grade Data v2_clean,GSCM 521 - Canvas Interaction Data v2_clean,random_student,random_student_name,618,0.739394,object,object,stringlike,stringlike,True,0.000000,0.000000,0.000000,0.000000,0.000000,0.284848,False,False,False,NaN,NaN,False,False,False,NaN,NaN,0.002238,0.997762,0.714773,0.285227,0.000000,right_join_key_not_unique_like,low,risky,M:M,Both sides have duplicates (neither side unique).
5,GSCM 521 - Canvas Gradebook Data_clean,GSCM 521 - Canvas Interaction Data v2_clean,random_student,random_student_name,618,0.739394,object,object,stringlike,stringlike,True,0.000000,0.000000,0.000000,0.000000,0.000000,0.284848,False,False,False,NaN,NaN,False,False,False,NaN,NaN,0.002238,0.997762,1.000000,0.000000,0.000000,right_join_key_not_unique_like,low,risky,1:M,Left looks key-like; right has duplicates (rig...
6,GSCM 521 - Canvas Interaction Data v2_clean,GSCM 521 - Banner Grade Data v2_clean,random_student_name,random_student,629,0.739394,object,object,stringlike,stringlike,True,0.000000,0.000000,0.000000,0.000000,0.000000,0.284848,False,False,False,NaN,NaN,False,False,False,NaN,NaN,0.714773,0.285227,0.002238,0.997762,0.000000,right_join_key_not_unique_like,low,risky,M:M,Both sides have duplicates (neither side unique).
7,GSCM 521 - Canvas Interaction Data v2_clean,GSCM 521 - Canvas Gradebook Data_clean,random_student_name,random_student,568,0.739394,object,object,stringlike,stringlike,True,0.000000,0.000000,0.000000,0.000000,0.000000,0.284848,False,False,False,NaN,NaN,False,False,False,NaN,NaN,1.000000,0.000000,0.002238,0.997762,0.000000,,low,ok,M:1,Right looks key-like; left has duplicates (lef...
8,GSCM 521 - Banner Grade Data v2_clean,GSCM 521 - Canvas Interaction Data v2_clean

In [10]:
# build join plan + write merge_plan sheet (expects recon cell already ran)

if "recon_enriched" not in globals() or recon_enriched is None or recon_enriched.empty:
    raise ValueError("recon_enriched is missing/empty. Run the recon generation cell first.")

plan = build_merge_plan(dfs, recon_enriched)

plan_proj = add_plan_projection(dfs, plan, rename_strategy="prefix")
plan_full = merge_plan_and_projection(plan, plan_proj)

plan_recon_review = build_plan_recon_review(plan_full, recon_enriched)

write_merge_plan_sheet(
    AUDIT_XLSX,
    plan_full,
    plan_recon_review,
    sheet_name="merge_plan",
    float_format="0.######",
    blank_rows_between_sections=2,
)

plan_full


,base_table,join_table,base_col,join_col,join_how,composite_score,evidence_max,name_similarity,base_dtype,join_dtype,base_null_rate,join_null_rate,right_n_nonnull,right_n_unique,right_unique_ratio,right_dup_nonnull,right_dup_rate,right_avg_matches_per_key,expected_match_rate,expected_null_after_left_join,sample_merge_inflation,explode_risk_score,explode_risk,coverage_risk,status,flag_reason,base_n_cols,right_n_cols,projected_added_cols,projected_total_cols_after
0,GSCM 521 - Canvas Interaction Data v2_clean,GSCM 521 - Canvas Gradebook Data_clean,sis_id,sis_id,left,0.998856,0.998239,1.000000,object,object,0.000000,0.000000,568,568,1.000000,0,0.000000,1.000000,0.998239,0.001761,0.000000,0.000000,low,low,ok,,15,3,2,17
1,GSCM 521 - Canvas Interaction Data v2_clean,GSCM 521 - Banner Grade Data v2_clean,random_student_name,random_student,left,0.284848,0.000000,0.739394,object,object,0.000000,0.000000,880,629,0.714773,251,0.285227,1.399046,0.000000,1.000000,0.000000,0.399046,high,high,flag,low_evidence_or_name_similarity;right_key_not_...,15,22,21,38


In [11]:
# human

resp = input("Execute joins now? Type YES to proceed: ").strip().upper()
if resp == "YES":
    joined = execute_merge_plan(dfs, plan, rename_strategy="prefix")
    display(joined.head(3))
else:
    print("Skipped executing joins.")


Execute joins now? Type YES to proceed:  


Skipped executing joins.


In [12]:
# audit/summary statistics (auto) -> write into merge_audit sheet blocks

base_table = plan_full.loc[0, "base_table"] if "plan_full" in globals() else plan.loc[0, "base_table"]
base_df = dfs[base_table]
joined_df = joined

auto_audit = post_join_audit(base_df, joined_df, top_null_columns=25)

auto_audit_blocks = [
    ("auto_row_integrity", auto_audit.get("row_integrity", pd.DataFrame())),
    ("auto_new_col_nulls_top", auto_audit.get("new_column_null_coverage_top", pd.DataFrame())),
]

# store for later use (and for manual append)
audit_blocks = auto_audit_blocks

# refresh merge_audit sheet so audits actually appear
if "recon_enriched" not in globals() or recon_enriched is None or recon_enriched.empty:
    raise ValueError("recon_enriched missing/empty. Run recon generation cell before writing merge_audit.")

if "diags" not in globals() or diags is None:
    diags = {}

write_merge_audit_sheet(
    AUDIT_XLSX,
    recon_enriched,
    diags,
    audit_blocks=audit_blocks,
    sheet_name="merge_audit",
    float_format="0.######",
    blank_cols_between=2,
    blank_rows_between_sections=2,
)

joined_df.shape


NameError: name 'joined' is not defined

# post interpretation / not automated

In [ ]:
# last-chance coverage check on flagged joins
# Run this for any join the plan flagged as "flag" or "no_candidate" before deciding to override.
#
# Pattern:
#   stats = left_join_coverage(
#       dfs["base_table_name"],
#       dfs["right_table_name"],
#       base_col="key_col_in_base",
#       join_col="key_col_in_right",
#   )
#   display(pd.DataFrame([stats]))
#
# A match_rate near 1.0 means the join key has full coverage — safe to override the flag.
# A low match_rate means genuine key misalignment — do not override without understanding why.
#
# Example (uncomment and edit):
# stats = left_join_coverage(
#     dfs["fact_table"],
#     dfs["dim_table"],
#     base_col="foreign_key_col",
#     join_col="primary_key_col",
# )
# display(pd.DataFrame([stats]))


In [ ]:
# force flagged joins — override plan status after manual review
# Only proceed if coverage check above confirms match_rate is acceptable.
#
# Edit join_table names to match your flagged tables.

resp = input("Proceed to manual/override joins now? Type YES to proceed: ").strip().upper()
if resp == "YES":
    # Override status for manually verified joins:
    # plan.loc[plan["join_table"].isin(["table_a", "table_b"]), "status"] = "ok"
    # plan.loc[plan["join_table"].isin(["table_a", "table_b"]), "flag_reason"] = "override: verified_full_key_coverage"
    
    joined = execute_merge_plan(dfs, plan, rename_strategy="prefix")
    display(joined.head(3))
else:
    print("Skipped manual/override joins.")
    raise SystemExit("Execution halted by user.")


In [ ]:
# audit/summary statistics (manual joins) -> append + refresh merge_audit sheet blocks

base_table = plan_full.loc[0, "base_table"] if "plan_full" in globals() else plan.loc[0, "base_table"]
base_df = dfs[base_table]
joined_df = joined

manu_audit = post_join_audit(base_df, joined_df, top_null_columns=25)

manu_audit_blocks = [
    ("manu_row_integrity", manu_audit.get("row_integrity", pd.DataFrame())),
    ("manu_new_col_nulls_top", manu_audit.get("new_column_null_coverage_top", pd.DataFrame())),
]

# append into audit_blocks if it exists; otherwise initialize
if "audit_blocks" in globals() and isinstance(audit_blocks, list):
    audit_blocks = audit_blocks + manu_audit_blocks
else:
    audit_blocks = manu_audit_blocks

write_merge_audit_sheet(
    AUDIT_XLSX,
    recon_enriched,
    diags,
    audit_blocks=audit_blocks,
    sheet_name="merge_audit",
    float_format="0.######",
    blank_cols_between=2,
    blank_rows_between_sections=2,
)

joined_df.shape


In [ ]:
# dataset-specific derived fields — add merge-produced columns here
#
# This is where you add fields computed from the joined data before export.
# Common patterns:
#   - Duration columns (end_date - start_date)
#   - Flag columns (e.g. drop_flag, status_flag based on null patterns)
#   - Composite keys built from joined columns
#   - Any normalization needed before analysis
#
# All columns added here should be documented in your project MD file.
#
# Pattern:
#   df_final = joined.copy()
#   df_final["new_col"] = ... compute from df_final ...
#   joined_final = df_final
#
# Example (commented out):
# df_final = joined.copy()
# df_final["duration_days"] = (
#     pd.to_datetime(df_final["end_date"], errors="coerce") -
#     pd.to_datetime(df_final["start_date"], errors="coerce")
# ).dt.days
# print(f"duration_days null count: {df_final['duration_days'].isna().sum():,}")
# joined_final = df_final

# If no derived fields are needed, just assign:
joined_final = joined.copy()


In [ ]:
# human
# Final export gate

resp = input("Write _merged_final outputs to CSV? Type YES to proceed: ").strip().upper()
if resp == "YES":
    os.makedirs(OUT_DIR, exist_ok=True)

    out_path = os.path.join(OUT_DIR, "_merged_final.csv")
    joined_final.to_csv(out_path, index=False)

    n_rows, n_cols = joined_final.shape
    file_size_mb = os.path.getsize(out_path) / (1024 * 1024)

    print("_merged_final.csv written successfully")
    print(f"Path: {out_path}")
    print(f"Rows: {n_rows:,}")
    print(f"Columns: {n_cols}")
    print(f"File size: {file_size_mb:.2f} MB")

    if n_cols > 0:
        print(f"Columns: {list(joined_final.columns)}")
else:
    print("Skipped writing outputs.")
    raise SystemExit("Execution halted by user.")
